FastText CC is the model of choice since its trained on generelized data instead of only tweets it got  articales blogs social media etc...


In [3]:
from pathlib import Path
import pandas as pd

# 1. Chargement des données nettoyées
DATA_DIR = Path(r"c:\Users\user\Projets_Ghoufrane\LLMS\Decetction-Persuation\data")
df_train = pd.read_json(DATA_DIR / "task1B_train_cleaned_augmented.jsonl", lines=True, encoding="utf-8")
df_val   = pd.read_json(DATA_DIR / "task1B_dev_cleaned.jsonl", lines=True, encoding="utf-8")
df_test  = pd.read_json(DATA_DIR / "task1B_test_cleaned.jsonl", lines=True, encoding="utf-8")


In [4]:
def compute_vocab_oov(train_texts, val_texts, test_texts):
    """
    Calcule la différence de vocabulaire (OOV) entre Train et Val, et entre Train et Test.
    """
    # Extraction des vocabulaires uniques
    vocab_train = set(w for text in train_texts if isinstance(text, str) for w in text.split())
    vocab_val   = set(w for text in val_texts if isinstance(text, str) for w in text.split())
    vocab_test  = set(w for text in test_texts if isinstance(text, str) for w in text.split())

    # Mots présents dans Val / Test mais absents du Train
    diff_val  = vocab_val - vocab_train
    diff_test = vocab_test - vocab_train

    return {
        "Train vs Val": {
            "total_vocab_val": len(vocab_val),
            "oov_count": len(diff_val),
            "oov_rate_%": round((len(diff_val) / len(vocab_val)) * 100, 2) if vocab_val else 0.0,
            "oov_words": diff_val
        },
        "Train vs Test": {
            "total_vocab_test": len(vocab_test),
            "oov_count": len(diff_test),
            "oov_rate_%": round((len(diff_test) / len(vocab_test)) * 100, 2) if vocab_test else 0.0,
            "oov_words": diff_test
        }
    }

In [5]:
results = compute_vocab_oov(df_train["text"], df_val["text"], df_test["text"])

# Affichage des résultats
print(f"OOV Train vs Val  : {results['Train vs Val']['oov_count']} mots ({results['Train vs Val']['oov_rate_%']}%)")
print(f"OOV Train vs Test : {results['Train vs Test']['oov_count']} mots ({results['Train vs Test']['oov_rate_%']}%)")

OOV Train vs Val  : 1677 mots (35.82%)
OOV Train vs Test : 3154 mots (43.55%)


# Tokenize BASIC

FastText constate que le mot exact n'est pas dans son dictionnaire.
Il découpe immédiatement ce mot inconnu en sous-mots : '<سي', 'سيك', 'يكت', 'كتب', 'تبو', 'بون', 'ون>'...
Il retrouve les vecteurs de ces fragments (qu'il connaît déjà grâce à d'autres mots comme سيارة, يكتب, يلعبون, etc.).
Il fait simplement la somme (ou la moyenne) des vecteurs de ses fragments
v
mot_inconnu
	​

=
g∈subwords
∑
	​

v
g
	​


Le vecteur ainsi créé n'est pas aléatoire : il hérite du sens sémantique de sa racine (كتب) et de ses préfixes/suffixes (سيـ, ـون).

In [6]:
def tokenize(text: str) -> list[str]:
    """Découpage basique sur les espaces pour texte déjà nettoyé."""
    return text.split() if isinstance(text, str) else []

In [7]:
for name, df in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
    # Ajout de la liste des tokens
    df["tokens"] = df["text"].apply(tokenize)
    # Ajout du nombre de mots par texte
    df["token_count"] = df["tokens"].apply(len)
    print(f"✅ {name:10} tokenisé ({len(df):,} exemples | {df['token_count'].sum():,} tokens au total)")
# 4. Aperçu du résultat sur le Train
print("\n--- Exemple de texte tokenisé (Train) ---")
display(df_train[["text", "tokens", "token_count"]].head(3))

✅ Train      tokenisé (2,535 exemples | 71,335 tokens au total)
✅ Validation tokenisé (259 exemples | 8,164 tokens au total)
✅ Test       tokenisé (503 exemples | 14,751 tokens au total)

--- Exemple de texte tokenisé (Train) ---


,text,tokens,token_count
0,جوجل يحتفل بعيد استقلال المملكة الأردنية الهاش...,"[جوجل, يحتفل, بعيد, استقلال, المملكة, الأردنية...",11
1,عاجل الأمن العراقي الهجوم تسبب باستشهاد آمر ال...,"[عاجل, الأمن, العراقي, الهجوم, تسبب, باستشهاد,...",19
2,وبحسب المنظمة الحقوقية الدولية، فقد دعت النياب...,"[وبحسب, المنظمة, الحقوقية, الدولية،, فقد, دعت,...",29


# charger le model

In [8]:
from huggingface_hub import hf_hub_download
import fasttext

model = fasttext.load_model(
    hf_hub_download("facebook/fasttext-ar-vectors", "model.bin")
)   

In [11]:
# 1. Vocabulaire unique du Train
vocab_train = set(w for text in df_train["text"] for w in text.split())

# 2. Vocabulaire du modèle officiel FastText
vocab_model = set(model.get_words())

# 3. Mots du Train absents du dictionnaire du modèle (OOV)
oov_train = vocab_train - vocab_model

# 4. Statistiques
oov_count = len(oov_train)
oov_rate = (oov_count / len(vocab_train)) * 100 if vocab_train else 0.0

print(f"Taille vocabulaire Train   : {len(vocab_train):,} mots")
print(f"Mots OOV (hors modèle)     : {oov_count:,} mots ({oov_rate:.2f}%)")
print(f"Couverture du modèle       : {100 - oov_rate:.2f}%")

Taille vocabulaire Train   : 22,414 mots
Mots OOV (hors modèle)     : 2,424 mots (10.81%)
Couverture du modèle       : 89.19%


In [13]:
# 1. Votre fonction tokenize (déjà présente dans votre notebook) :
# def tokenize(text: str) -> list[str]:
#     return text.split() if isinstance(text, str) else []

# 2. Application sur l'ensemble des splits : Train, Val et Test
splits = {
    "Train": df_train,
    "Val": df_val,
}

for name, df in splits.items():
    # A. Liste des tokens (mots découpés)
    df["tokens"] = df["text"].apply(tokenize)
    
    # C. Nombre de tokens par texte
    df["token_count"] = df["tokens"].apply(len)
    
    print(f"✅ {name:5} : {len(df):,} exemples | {df['token_count'].sum():,} tokens")

# 3. Aperçu rapide
display(df_train[["text", "tokens", "token_count"]].head(2))

✅ Train : 2,535 exemples | 71,335 tokens
✅ Val   : 259 exemples | 8,164 tokens


,text,tokens,token_count
0,جوجل يحتفل بعيد استقلال المملكة الأردنية الهاش...,"[جوجل, يحتفل, بعيد, استقلال, المملكة, الأردنية...",11
1,عاجل الأمن العراقي الهجوم تسبب باستشهاد آمر ال...,"[عاجل, الأمن, العراقي, الهجوم, تسبب, باستشهاد,...",19


# Embed_data

    Transforme une liste de textes tokenisés en représentations vectorielles FastText.

    Paramètres à ajuster (Tweaks) :
    --------------------------------
    :param tokenized_texts: Liste de listes de tokens (ex: df["tokens"].tolist()).
    :param model: Modèle FastText chargé (ex: model = fasttext.load_model(...)).
    :param mode: Type de sortie :
        - 'sequence' : Tenseur 3D (N, max_len, dim) pour BiLSTM, CNN, Transformer.
        - 'mean'     : Matrice 2D (N, dim) - moyenne des mots de la phrase.
        - 'max'      : Matrice 2D (N, dim) - maximum dimension par dimension.
        - 'mean_max' : Matrice 2D (N, dim * 2) - concaténation moyenne + max pooling.
    :param max_len: Longueur fixe des séquences (utilisé uniquement en mode 'sequence').
    :param padding: 'post' (zéros ajoutés à la fin) ou 'pre' (zéros ajoutés au début).
    :param truncating: 'post' (tronque les mots de la fin) ou 'pre' (tronque le début).
    :param use_subwords: 
        - True  : FastText calcule un vecteur pour TOUS les mots via ses sous-mots.
        - False : N'utilise que les mots du dictionnaire exact ; les autres deviennent UNK.
    :param unk_strategy: 'mean' (vecteur moyen neutre), 'random' ou 'zeros'.
    :param show_progress: Affiche la barre tqdm.

    :return: np.ndarray (2D ou 3D selon le mode).

Mean Pooling : effectue la moyenne point par point de tous les mots du texte pour résumer la phrase entière en un seul vecteur représentatif de 300 dimensions.

Empile verticalement tous les vecteurs individuels pour créer une matrice 2D exploitable directement par vos classifieurs ou modèles de Deep Learning (shape = (nb_lignes, 300)).

In [ ]:
import numpy as np
from tqdm.auto import tqdm

def get_unk_vector(model, strategy: str = "mean", dim: int = 300) -> np.ndarray:
    """Calcule le vecteur de secours pour les mots totalement inconnus."""
    if strategy == "mean":
        # Moyenne d'un échantillon représentatif du modèle (neutre sémantiquement)
        sample_words = model.get_words()[:10000]
        sample_vecs = np.array([model.get_word_vector(w) for w in sample_words])
        return np.mean(sample_vecs, axis=0).astype(np.float32)
    elif strategy == "random":
        # Vecteur gaussien à faible variance
        return np.random.normal(0.0, 0.05, size=(dim,)).astype(np.float32)
    else:
        return np.zeros(dim, dtype=np.float32)


def embed_data(
    tokenized_texts: list[list[str]],
    model,
    mode: str = "sequence",
    max_len: int = 100,
    padding: str = "post",
    truncating: str = "post",
    use_subwords: bool = True,
    unk_strategy: str = "mean",
    show_progress: bool = True
) -> np.ndarray:

    dim = model.get_dimension()
    pad_vector = np.zeros(dim, dtype=np.float32)  # Règle d'or pour le padding
    unk_vector = get_unk_vector(model, strategy=unk_strategy, dim=dim)
    words_set = set(model.get_words()) if not use_subwords else None

    # Fonction pour convertir un token en vecteur
    def get_token_vector(token: str) -> np.ndarray:
        if not isinstance(token, str) or not token.strip():
            return unk_vector
        if use_subwords:
            return model.get_word_vector(token).astype(np.float32)
        else:
            return model.get_word_vector(token).astype(np.float32) if token in words_set else unk_vector

    all_embeddings = []
    iterator = tqdm(tokenized_texts, desc=f"Embedding ({mode})") if show_progress else tokenized_texts

    for tokens in iterator:
        # Récupération des vecteurs des mots du texte
        word_vectors = [get_token_vector(t) for t in tokens] if len(tokens) > 0 else [unk_vector]

        # -----------------------------------------------------------------
        # Option A : Mode Séquence 3D (pour BiLSTM, CNN, etc.)
        # -----------------------------------------------------------------
        if mode == "sequence":
            # 1. Troncature si le texte dépasse max_len
            if len(word_vectors) > max_len:
                seq_vectors = word_vectors[:max_len] if truncating == "post" else word_vectors[-max_len:]
            else:
                seq_vectors = word_vectors

            # 2. Padding avec zéros si le texte est plus court que max_len
            pad_count = max_len - len(seq_vectors)
            if pad_count > 0:
                pads = [pad_vector] * pad_count
                seq_vectors = (seq_vectors + pads) if padding == "post" else (pads + seq_vectors)

            all_embeddings.append(np.array(seq_vectors, dtype=np.float32))

        # -----------------------------------------------------------------
        # Option B : Mode Pooling 2D (pour classifieurs classiques / LogisticRegression)
        # -----------------------------------------------------------------
        elif mode == "mean":
            all_embeddings.append(np.mean(word_vectors, axis=0).astype(np.float32))

        elif mode == "max":
            all_embeddings.append(np.max(word_vectors, axis=0).astype(np.float32))

        elif mode == "mean_max":
            mean_vec = np.mean(word_vectors, axis=0).astype(np.float32)
            max_vec  = np.max(word_vectors, axis=0).astype(np.float32)
            all_embeddings.append(np.concatenate([mean_vec, max_vec]))

    return np.array(all_embeddings, dtype=np.float32)